## Class Survey

This notebook loads our class survey responses from Google Sheets and visualizes them with bar charts, a histogram, and word clouds — using only `datascience` Tables for all data work. The plotting logic lives in `survey_helpers.py` alongside this notebook; open that file if you want to see how a function works.

### 1. Setup

In [ ]:
from datascience import *
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from survey_helpers import (
    load_survey, plot_bar_chart, plot_histogram, plot_wordcloud, auto_plot_all,
    standardize_categorical, numeric_summary
)


### 2. Load the survey data

This pulls the responses straight from the Google Sheet as a CSV export into a `Table`.

In [ ]:
sheet_id = '1uO4vrzGn5uPPHJfrbTA4kplefMzzLXAHWz37tgOoiKI'
gid = '1857465765'

survey_table = load_survey(sheet_id, gid)
survey_table.show()


### 3. Filtering by Course Section

`group` gives us a quick count of how many responses came from each section.

In [ ]:
survey_table.group('Course Section')


Use `.where(...)` to select just one section. Swap `'001'` below for whichever section label shows up in the table above, then re-run the cells in the rest of this notebook with `section_table` instead of `survey_table` to look at just that section.

In [ ]:
section_table = survey_table.where('Course Section', are.equal_to('001'))
section_table.show()


### 4. Course Section

A bar chart showing how many respondents are in each section.

In [ ]:
plot_bar_chart(survey_table, 'Course Section')


### 5. Where do students hail from?

Free-text answers like this one are messy: the same city can show up as `'Philadelphia'`, `'philadelphia '`, or `'  PHILADELPHIA'`. If we `.group(...)` the raw column, each of those variations gets counted as its own separate category.

In [ ]:
hometown_column = 'Where do you hail from? (Translation: Where do you consider home)'

# Raw counts -- extra spaces and inconsistent capitalization split what should be
# the same answer into multiple rows
survey_table.group(hometown_column).sort('count', descending=True)


`standardize_categorical` strips extra whitespace and normalizes capitalization to Title Case, so `'philadelphia '` and `'Philadelphia'` collapse into a single answer. This same function works on any free-text column -- try it on `'Current Temple major'` or `'What is your favorite late night snack?'` too.

In [ ]:
cleaned_hometowns = standardize_categorical(survey_table, hometown_column)

# Cleaned counts -- matching responses are now grouped together
cleaned_hometowns.group(hometown_column).sort('count', descending=True)


With the column cleaned up, a word cloud gives us a quick visual sense of where people are from.

In [ ]:
plot_wordcloud(cleaned_hometowns, hometown_column, title='Where do you hail from?')


### 6. Superpowers

Another free-text question — word cloud again.

In [ ]:
plot_wordcloud(survey_table, 'What is you Super power?', title='Superpowers')


### 7. Favorite late night snack

Same idea for snacks.

In [ ]:
plot_wordcloud(survey_table, 'What is your favorite late night snack?', title='Favorite Late Night Snack')


### 8. Coding experience rating

This is a numeric rating, so a histogram makes more sense than a word cloud. `numeric_summary` also gives us the usual descriptive statistics -- count, mean, median, standard deviation, min, and max -- as a `Table`.

In [ ]:
numeric_summary(survey_table, 'Rate your experience coding')


In [ ]:
plot_histogram(survey_table, 'Rate your experience coding', bins=5,
                title='Rate Your Experience Coding')


We can reuse `section_table` (from step 3) to compare one section's ratings against the whole class.

In [ ]:
numeric_summary(section_table, 'Rate your experience coding')


### 9. Current Temple major

Bar chart of declared majors.

In [ ]:
plot_bar_chart(survey_table, 'Current Temple major', top_n=20)


### 10. Anticipated graduation year

Bar chart of anticipated graduation years.

In [ ]:
plot_bar_chart(survey_table, 'Anticipated graduation year', top_n=10)


### 11. Comparing sections

Because every plotting function takes a `Table` as its first argument, you can pass in `section_table` (from step 3) instead of `survey_table` to see the same charts for just one section — handy for comparing sections side by side.

In [ ]:
plot_bar_chart(section_table, 'Current Temple major', top_n=20,
                title='Current Temple Major (Section 001)')


### 12. (Optional) Auto-plot every question

`auto_plot_all` loops over every column in a Table (skipping `Timestamp`) and picks a chart type automatically:
- **Mostly-numeric** columns with few unique values → histogram
- **Text columns with few unique values** relative to responses → bar chart (with whitespace/capitalization cleanup applied automatically)
- **Everything else** (long, varied free text) → word cloud

Useful if new questions get added to the sheet later and you don't want to write a new cell for each one.

In [ ]:
# Uncomment to run on the whole class, or pass section_table to run on one section:
# auto_plot_all(survey_table)
